# 🚗 Automotive Car Price Estimation — Exploratory Data Analysis

**Masterschool Data Science Project**  
**Author:** Hande Gabrali-Knobloch  
**Date:** 2024

---

This notebook covers:
1. Data loading and overview
2. Descriptive statistics
3. Target variable analysis (price distribution)
4. Univariate analysis of categorical features
5. Univariate analysis of numeric features
6. Bivariate analysis (price vs features)
7. Correlation analysis
8. Feature engineering overview
9. Key insights and next steps

In [ ]:
# ── 1. Imports ──────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import sys
sys.path.insert(0, '..')
from src import load_sample_data, clean_data, engineer_features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded successfully')

In [ ]:
# ── 2. Load Data ─────────────────────────────────────────────────
# Use synthetic data for demonstration; replace with load_data('data/raw/car_listings.csv')
df_raw = load_sample_data()
print(f'Raw shape: {df_raw.shape}')
df_raw.head(10)

In [ ]:
# ── 3. Basic Statistics ──────────────────────────────────────────
print('=== Data Types ===')
print(df_raw.dtypes)
print('\n=== Missing Values ===')
print(df_raw.isnull().sum())
print('\n=== Numeric Summary ===')
df_raw.describe().round(2)

In [ ]:
# ── 4. Clean & Engineer Features ─────────────────────────────────
df = clean_data(df_raw)
df = engineer_features(df)
print(f'Cleaned shape: {df.shape}')
print(f'New features: {[c for c in df.columns if c not in df_raw.columns]}')

In [ ]:
# ── 5. Target Distribution ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['price'].median(), color='red', linestyle='--', label=f"Median: ${df['price'].median():,.0f}")
axes[0].set_title('Car Price Distribution', fontsize=14)
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(np.log1p(df['price']), bins=50, color='teal', edgecolor='white', alpha=0.85)
axes[1].set_title('log(1 + Price) Distribution', fontsize=14)
axes[1].set_xlabel('log(1 + Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../reports/figures/price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Price stats: min=${df['price'].min():,} | median=${df['price'].median():,} | mean=${df['price'].mean():,.0f} | max=${df['price'].max():,}")

In [ ]:
# ── 6. Categorical Feature Analysis ─────────────────────────────
cat_cols = ['make', 'fuel_type', 'transmission', 'body_style', 'drive_type', 'condition']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, col in zip(axes.flat, cat_cols):
    order = df.groupby(col)['price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='price', order=order, palette='muted', ax=ax)
    ax.set_title(f'Price by {col.replace("_", " ").title()}', fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('../reports/figures/price_by_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7. Numeric Feature Analysis ──────────────────────────────────
num_cols = ['year', 'mileage', 'horsepower', 'engine_size', 'age_years', 'mileage_per_year']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, col in zip(axes.flat, num_cols):
    ax.scatter(df[col], df['price'], alpha=0.3, s=8, color='steelblue')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Price ($)')
    ax.set_title(f'{col.replace("_", " ").title()} vs Price', fontsize=12)

    # Add trend line
    z = np.polyfit(df[col].dropna(), df.loc[df[col].notna(), 'price'], 1)
    p = np.poly1d(z)
    x_sorted = np.sort(df[col].dropna())
    ax.plot(x_sorted, p(x_sorted), 'r--', linewidth=1.5)

plt.tight_layout()
plt.savefig('../reports/figures/numeric_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. Correlation Heatmap ───────────────────────────────────────
num_df = df.select_dtypes(include=[np.number]).drop(columns=['is_luxury', 'high_mileage'], errors='ignore')
corr = num_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, square=True, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=15, pad=20)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top correlations with price:')
print(corr['price'].sort_values(ascending=False).drop('price').head(10))

In [ ]:
# ── 9. Key Insights Summary ──────────────────────────────────────
print('=' * 60)
print('KEY INSIGHTS FROM EDA')
print('=' * 60)

insights = [
    f"1. Dataset: {len(df):,} records, {df.shape[1]} features after engineering",
    f"2. Price range: ${df['price'].min():,} - ${df['price'].max():,}",
    f"3. Median price: ${df['price'].median():,}",
    f"4. Year range: {df['year'].min()} - {df['year'].max()}",
    f"5. Avg mileage: {df['mileage'].mean():,.0f} km",
    f"6. Luxury cars: {df['is_luxury'].sum()} ({df['is_luxury'].mean()*100:.1f}%)",
    f"7. High-mileage: {df['high_mileage'].sum()} ({df['high_mileage'].mean()*100:.1f}%)",
    "8. Price is right-skewed — log transform recommended",
    "9. Strongest predictors: age_years, mileage, horsepower",
    "10. Luxury brand premium: ~30-50% above comparable non-luxury",
]

for insight in insights:
    print(insight)

print('\nNext Steps:')
print('  → Run notebooks/02_Feature_Engineering.ipynb')
print('  → Run notebooks/03_Model_Training.ipynb')